# 06 — Revenue Trend Forecasting
Fits ARIMA forecasts for the 20 largest companies with sufficient non-TTM history.

In [ ]:
from pathlib import Path
import warnings, pandas as pd
from statsmodels.tsa.arima.model import ARIMA
root=Path.cwd(); pl=pd.read_csv(root/'data'/'clean'/'profitandloss.csv'); pl=pl.loc[~pl.is_ttm.astype(bool)].sort_values('sort_order'); top=pl.groupby('symbol').tail(1).nlargest(20,'sales').symbol; forecasts=[]
for symbol in top:
 series=pl.loc[pl.symbol.eq(symbol),'sales'].dropna().astype(float)
 if len(series)<6: continue
 try:
  with warnings.catch_warnings(): warnings.simplefilter('ignore'); fitted=ARIMA(series,order=(1,1,1)).fit(); estimate=float(fitted.forecast(1).iloc[0])
  forecasts.append({'symbol':symbol,'last_sales':series.iloc[-1],'forecast_sales_1y':estimate,'growth_pct':(estimate/series.iloc[-1]-1)*100})
 except Exception as exc: print(symbol,exc)
result=pd.DataFrame(forecasts); out=root/'data'/'warehouse'/'revenue_forecasts.csv'; result.to_csv(out,index=False); display(result.sort_values('growth_pct',ascending=False)); print(out)